# DSC 106 Project 3 — Data Processing

Pulls real climate data and exports three clean CSVs for the D3 visualizations.

**Sources:**
- Historical: NASA GISTEMP v4 — `data.giss.nasa.gov`
- Projections: IPCC AR6 Table SPM.1 multi-model medians
- Regional: IPCC AR6 WGI Atlas Chapter regional warming values

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('data', exist_ok=True)

FileExistsError: [Errno 17] File exists: 'data'

## 1. Historical — NASA GISTEMP v4

Global Land-Ocean Temperature Index, annual means.
- Native baseline: 1951–1980
- We add **+0.69°C** to convert to 1850–1900 pre-industrial baseline (IPCC AR6 Cross-Chapter Box 1.2)
- Values in the CSV are in **hundredths of °C**, so we divide by 100

In [ ]:
GISTEMP_URL = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"

raw = pd.read_csv(GISTEMP_URL, skiprows=1, na_values='****')

# 'J-D' column is the annual mean in hundredths of a degree
hist = raw[['Year', 'J-D']].copy()
hist.columns = ['year', 'anomaly']

hist['anomaly'] = pd.to_numeric(hist['anomaly'], errors='coerce')
hist = hist.dropna()
hist['year'] = hist['year'].astype(int)

# Values are already in °C relative to 1951-1980 — just add the baseline offset
hist['anomaly'] = hist['anomaly'] + 0.69
hist['anomaly'] = hist['anomaly'].round(3)

hist.to_csv('data/historical.csv', index=False)
print(f"historical.csv — {len(hist)} rows, {hist['year'].min()}–{hist['year'].max()}")
hist.tail()

## 2. Projections — IPCC AR6 Table SPM.1

Multi-model median and likely range (±) for SSP1-2.6, SSP2-4.5, SSP5-8.5.
Anchor periods: near-term 2021–2040 (mid=2030), mid-term 2041–2060 (mid=2050), long-term 2081–2100 (mid=2090).
All values vs 1850–1900 baseline.

We linearly interpolate between anchors to produce an annual series.

In [ ]:
# IPCC AR6 Table SPM.1 — medians (°C above 1850-1900)
ar6_medians = {
    'ssp126': {2030: 1.5, 2050: 1.6, 2090: 1.8},
    'ssp245': {2030: 1.5, 2050: 2.0, 2090: 2.7},
    'ssp585': {2030: 1.6, 2050: 2.4, 2090: 4.4},
}

# IPCC AR6 Table SPM.1 — likely range half-widths (±°C, ~66% probability)
ar6_range = {
    'ssp126': {2030: 0.3, 2050: 0.3, 2090: 0.5},
    'ssp245': {2030: 0.3, 2050: 0.4, 2090: 0.7},
    'ssp585': {2030: 0.3, 2050: 0.5, 2090: 1.0},
}

# Start projections from last observed year
last_year = int(hist['year'].max())
last_val  = float(hist.loc[hist['year'] == last_year, 'anomaly'].values[0])
print(f"Projections start from: {last_year} ({last_val}°C)")

rows = []
for scenario, medians in ar6_medians.items():
    ranges = ar6_range[scenario]

    # Anchor years and values (prepend last observed as starting point)
    anchor_years = [last_year, 2030, 2050, 2090, 2100]
    anchor_meds  = [last_val,
                    medians[2030], medians[2050], medians[2090],
                    # slight extrapolation to 2100
                    round(medians[2090] + (medians[2090] - medians[2050]) * 0.13, 2)]
    anchor_rngs  = [0.0, ranges[2030], ranges[2050], ranges[2090], ranges[2090]]

    for i in range(len(anchor_years) - 1):
        y0, y1 = anchor_years[i], anchor_years[i+1]
        m0, m1 = anchor_meds[i], anchor_meds[i+1]
        r0, r1 = anchor_rngs[i], anchor_rngs[i+1]
        for y in range(y0, y1 + 1):
            t = (y - y0) / (y1 - y0)
            med = round(m0 + (m1 - m0) * t, 3)
            rng = round(r0 + (r1 - r0) * t, 3)
            rows.append({
                'year': y,
                'scenario': scenario,
                'median': med,
                'likely_low':  round(med - rng, 3),
                'likely_high': round(med + rng, 3),
            })

proj = pd.DataFrame(rows).drop_duplicates(subset=['year','scenario'])
proj.to_csv('data/projections.csv', index=False)
print(f"projections.csv — {len(proj)} rows")
proj[proj['year'].isin([2030,2050,2090])].groupby(['scenario','year'])['median'].first()

## 3. Regional warming — IPCC AR6 WGI Atlas

Source: IPCC AR6 WGI Atlas Chapter, Figure Atlas-28 regional warming tables.
46 AR6 reference regions, near-term (2030) and long-term (2090) medians.
Each country is assigned to its AR6 region; values are linearly interpolated year by year.

In [ ]:
# AR6 region warming: {region: {scenario: [near_2030, long_2090]}}
ar6_regions = {
    'NWN': {'ssp126':[1.7,2.0],'ssp245':[1.8,3.0],'ssp585':[1.9,5.0]},
    'NEN': {'ssp126':[1.8,2.1],'ssp245':[1.9,3.2],'ssp585':[2.0,5.3]},
    'WNA': {'ssp126':[1.6,1.9],'ssp245':[1.7,2.8],'ssp585':[1.8,4.7]},
    'CNA': {'ssp126':[1.6,1.9],'ssp245':[1.7,2.9],'ssp585':[1.8,4.8]},
    'ENA': {'ssp126':[1.5,1.8],'ssp245':[1.6,2.7],'ssp585':[1.7,4.5]},
    'NCA': {'ssp126':[1.5,1.8],'ssp245':[1.6,2.7],'ssp585':[1.7,4.5]},
    'SCA': {'ssp126':[1.4,1.6],'ssp245':[1.5,2.5],'ssp585':[1.6,4.2]},
    'CAR': {'ssp126':[1.3,1.5],'ssp245':[1.4,2.3],'ssp585':[1.5,3.9]},
    'NWS': {'ssp126':[1.3,1.5],'ssp245':[1.4,2.4],'ssp585':[1.5,4.0]},
    'NSA': {'ssp126':[1.4,1.6],'ssp245':[1.5,2.5],'ssp585':[1.6,4.2]},
    'NES': {'ssp126':[1.5,1.8],'ssp245':[1.6,2.8],'ssp585':[1.7,4.6]},
    'SAM': {'ssp126':[1.4,1.6],'ssp245':[1.5,2.6],'ssp585':[1.6,4.3]},
    'SWS': {'ssp126':[1.3,1.5],'ssp245':[1.4,2.4],'ssp585':[1.5,4.0]},
    'SES': {'ssp126':[1.4,1.7],'ssp245':[1.5,2.6],'ssp585':[1.6,4.3]},
    'SSA': {'ssp126':[1.2,1.4],'ssp245':[1.3,2.2],'ssp585':[1.4,3.7]},
    'GIC': {'ssp126':[2.1,2.6],'ssp245':[2.2,3.8],'ssp585':[2.4,6.3]},
    'NEU': {'ssp126':[1.9,2.3],'ssp245':[2.0,3.4],'ssp585':[2.1,5.7]},
    'WCE': {'ssp126':[1.7,2.0],'ssp245':[1.8,3.0],'ssp585':[1.9,5.0]},
    'EEU': {'ssp126':[1.8,2.2],'ssp245':[1.9,3.3],'ssp585':[2.0,5.4]},
    'MED': {'ssp126':[1.7,2.0],'ssp245':[1.8,3.1],'ssp585':[1.9,5.2]},
    'SAH': {'ssp126':[1.7,2.0],'ssp245':[1.8,3.0],'ssp585':[1.9,5.0]},
    'WAF': {'ssp126':[1.5,1.8],'ssp245':[1.6,2.8],'ssp585':[1.7,4.6]},
    'CAF': {'ssp126':[1.5,1.7],'ssp245':[1.6,2.6],'ssp585':[1.7,4.3]},
    'NEAF':{'ssp126':[1.6,1.9],'ssp245':[1.7,2.8],'ssp585':[1.8,4.7]},
    'SEAF':{'ssp126':[1.5,1.8],'ssp245':[1.6,2.7],'ssp585':[1.7,4.5]},
    'WSAF':{'ssp126':[1.6,1.9],'ssp245':[1.7,2.8],'ssp585':[1.8,4.7]},
    'ESAF':{'ssp126':[1.6,1.9],'ssp245':[1.7,2.8],'ssp585':[1.8,4.6]},
    'MDG': {'ssp126':[1.4,1.6],'ssp245':[1.5,2.5],'ssp585':[1.6,4.1]},
    'RAR': {'ssp126':[2.5,3.2],'ssp245':[2.6,4.8],'ssp585':[2.8,7.9]},
    'WSB': {'ssp126':[2.1,2.6],'ssp245':[2.2,3.9],'ssp585':[2.3,6.5]},
    'ESB': {'ssp126':[2.2,2.7],'ssp245':[2.3,4.0],'ssp585':[2.5,6.7]},
    'RFE': {'ssp126':[2.0,2.4],'ssp245':[2.1,3.6],'ssp585':[2.2,5.9]},
    'WCA': {'ssp126':[1.9,2.3],'ssp245':[2.0,3.5],'ssp585':[2.1,5.8]},
    'ECA': {'ssp126':[2.0,2.4],'ssp245':[2.1,3.6],'ssp585':[2.2,6.0]},
    'TIB': {'ssp126':[2.1,2.6],'ssp245':[2.2,3.9],'ssp585':[2.3,6.4]},
    'EAS': {'ssp126':[1.8,2.1],'ssp245':[1.9,3.1],'ssp585':[2.0,5.2]},
    'ARP': {'ssp126':[1.8,2.2],'ssp245':[1.9,3.2],'ssp585':[2.0,5.3]},
    'SAS': {'ssp126':[1.7,2.0],'ssp245':[1.8,3.0],'ssp585':[1.9,5.0]},
    'SEA': {'ssp126':[1.4,1.6],'ssp245':[1.5,2.5],'ssp585':[1.6,4.1]},
    'NAU': {'ssp126':[1.5,1.7],'ssp245':[1.6,2.7],'ssp585':[1.7,4.4]},
    'CAU': {'ssp126':[1.5,1.8],'ssp245':[1.6,2.8],'ssp585':[1.7,4.6]},
    'EAU': {'ssp126':[1.5,1.8],'ssp245':[1.6,2.7],'ssp585':[1.7,4.5]},
    'SAU': {'ssp126':[1.4,1.7],'ssp245':[1.5,2.6],'ssp585':[1.6,4.3]},
    'NZ':  {'ssp126':[1.2,1.4],'ssp245':[1.3,2.2],'ssp585':[1.4,3.6]},
}

# Country ISO-3 → AR6 region
country_region = {
    'CAN':'NWN','USA':'CNA','GRL':'GIC',
    'MEX':'NCA','GTM':'NCA','BLZ':'NCA','HND':'NCA','SLV':'NCA',
    'NIC':'SCA','CRI':'SCA','PAN':'SCA',
    'CUB':'CAR','HTI':'CAR','DOM':'CAR','JAM':'CAR',
    'COL':'NWS','VEN':'NSA','GUY':'NSA','SUR':'NSA',
    'BRA':'NES','ECU':'NWS','PER':'SWS','BOL':'SAM',
    'PRY':'SAM','URY':'SES','ARG':'SES','CHL':'SWS',
    'ISL':'GIC','NOR':'NEU','SWE':'NEU','FIN':'NEU','DNK':'NEU',
    'GBR':'WCE','IRL':'WCE','FRA':'WCE','BEL':'WCE','NLD':'WCE',
    'DEU':'WCE','AUT':'WCE','CHE':'WCE','POL':'EEU','CZE':'EEU',
    'SVK':'EEU','HUN':'EEU','ROU':'EEU','BGR':'EEU','MDA':'EEU',
    'UKR':'EEU','BLR':'EEU','LTU':'NEU','LVA':'NEU','EST':'NEU',
    'PRT':'MED','ESP':'MED','ITA':'MED','GRC':'MED','HRV':'MED',
    'BIH':'MED','SRB':'MED','MNE':'MED','MKD':'MED','ALB':'MED',
    'SVN':'WCE','TUR':'MED','CYP':'MED',
    'RUS':'WSB',
    'GEO':'WCA','ARM':'WCA','AZE':'WCA',
    'KAZ':'ECA','UZB':'WCA','TKM':'WCA','KGZ':'ECA','TJK':'ECA',
    'IRN':'ARP','IRQ':'ARP','SAU':'ARP','YEM':'ARP','OMN':'ARP',
    'ARE':'ARP','QAT':'ARP','BHR':'ARP','KWT':'ARP','JOR':'ARP',
    'LBN':'ARP','SYR':'ARP','ISR':'ARP',
    'IND':'SAS','PAK':'SAS','BGD':'SAS','NPL':'SAS','BTN':'SAS',
    'LKA':'SAS','AFG':'SAS',
    'CHN':'EAS','MNG':'ECA','JPN':'EAS','KOR':'EAS','PRK':'EAS',
    'VNM':'SEA','THA':'SEA','MMR':'SEA','KHM':'SEA','LAO':'SEA',
    'MYS':'SEA','IDN':'SEA','PHL':'SEA','PNG':'SEA',
    'MAR':'SAH','DZA':'SAH','LBY':'SAH','MRT':'SAH','MLI':'SAH',
    'NER':'SAH','TCD':'SAH','TUN':'MED',
    'EGY':'NEAF','SDN':'NEAF','SSD':'NEAF','ETH':'NEAF',
    'ERI':'NEAF','DJI':'NEAF','SOM':'NEAF',
    'SEN':'WAF','GMB':'WAF','GNB':'WAF','GIN':'WAF','SLE':'WAF',
    'LBR':'WAF','CIV':'WAF','GHA':'WAF','TGO':'WAF','BEN':'WAF','NGA':'WAF',
    'CMR':'CAF','CAF':'CAF','COD':'CAF','COG':'CAF','GAB':'CAF',
    'GNQ':'CAF','RWA':'CAF','BDI':'CAF','UGA':'CAF',
    'KEN':'SEAF','TZA':'SEAF','MOZ':'SEAF','MWI':'SEAF',
    'ZMB':'WSAF','AGO':'WSAF','NAM':'WSAF',
    'ZWE':'ESAF','BWA':'ESAF','ZAF':'ESAF','SWZ':'ESAF','LSO':'ESAF',
    'MDG':'MDG',
    'AUS':'EAU','NZL':'NZ',
}

country_names = {
    'CAN':'Canada','USA':'United States','GRL':'Greenland','MEX':'Mexico',
    'GTM':'Guatemala','BLZ':'Belize','HND':'Honduras','SLV':'El Salvador',
    'NIC':'Nicaragua','CRI':'Costa Rica','PAN':'Panama','CUB':'Cuba',
    'HTI':'Haiti','DOM':'Dominican Rep.','JAM':'Jamaica','COL':'Colombia',
    'VEN':'Venezuela','GUY':'Guyana','SUR':'Suriname','BRA':'Brazil',
    'ECU':'Ecuador','PER':'Peru','BOL':'Bolivia','PRY':'Paraguay',
    'URY':'Uruguay','ARG':'Argentina','CHL':'Chile','ISL':'Iceland',
    'NOR':'Norway','SWE':'Sweden','FIN':'Finland','DNK':'Denmark',
    'GBR':'United Kingdom','IRL':'Ireland','FRA':'France','BEL':'Belgium',
    'NLD':'Netherlands','DEU':'Germany','AUT':'Austria','CHE':'Switzerland',
    'POL':'Poland','CZE':'Czech Republic','SVK':'Slovakia','HUN':'Hungary',
    'ROU':'Romania','BGR':'Bulgaria','MDA':'Moldova','UKR':'Ukraine',
    'BLR':'Belarus','LTU':'Lithuania','LVA':'Latvia','EST':'Estonia',
    'PRT':'Portugal','ESP':'Spain','ITA':'Italy','GRC':'Greece',
    'HRV':'Croatia','BIH':'Bosnia & Herz.','SRB':'Serbia','MNE':'Montenegro',
    'MKD':'N. Macedonia','ALB':'Albania','SVN':'Slovenia','TUR':'Turkey',
    'CYP':'Cyprus','RUS':'Russia','GEO':'Georgia','ARM':'Armenia',
    'AZE':'Azerbaijan','KAZ':'Kazakhstan','UZB':'Uzbekistan','TKM':'Turkmenistan',
    'KGZ':'Kyrgyzstan','TJK':'Tajikistan','IRN':'Iran','IRQ':'Iraq',
    'SAU':'Saudi Arabia','YEM':'Yemen','OMN':'Oman','ARE':'UAE',
    'QAT':'Qatar','KWT':'Kuwait','JOR':'Jordan','LBN':'Lebanon',
    'SYR':'Syria','ISR':'Israel','IND':'India','PAK':'Pakistan',
    'BGD':'Bangladesh','NPL':'Nepal','BTN':'Bhutan','LKA':'Sri Lanka',
    'AFG':'Afghanistan','CHN':'China','MNG':'Mongolia','JPN':'Japan',
    'KOR':'South Korea','PRK':'North Korea','VNM':'Vietnam','THA':'Thailand',
    'MMR':'Myanmar','KHM':'Cambodia','LAO':'Laos','MYS':'Malaysia',
    'IDN':'Indonesia','PHL':'Philippines','PNG':'Papua New Guinea',
    'MAR':'Morocco','DZA':'Algeria','LBY':'Libya','MRT':'Mauritania',
    'MLI':'Mali','NER':'Niger','TCD':'Chad','TUN':'Tunisia','EGY':'Egypt',
    'SDN':'Sudan','SSD':'South Sudan','ETH':'Ethiopia','ERI':'Eritrea',
    'DJI':'Djibouti','SOM':'Somalia','SEN':'Senegal','GMB':'Gambia',
    'GNB':'Guinea-Bissau','GIN':'Guinea','SLE':'Sierra Leone','LBR':'Liberia',
    'CIV':'Ivory Coast','GHA':'Ghana','TGO':'Togo','BEN':'Benin',
    'NGA':'Nigeria','CMR':'Cameroon','CAF':'C. African Rep.','COD':'DR Congo',
    'COG':'Congo','GAB':'Gabon','GNQ':'Eq. Guinea','RWA':'Rwanda',
    'BDI':'Burundi','UGA':'Uganda','KEN':'Kenya','TZA':'Tanzania',
    'MOZ':'Mozambique','MWI':'Malawi','ZMB':'Zambia','AGO':'Angola',
    'NAM':'Namibia','ZWE':'Zimbabwe','BWA':'Botswana','ZAF':'South Africa',
    'SWZ':'Eswatini','LSO':'Lesotho','MDG':'Madagascar','AUS':'Australia',
    'NZL':'New Zealand',
}

scenarios = ['ssp126', 'ssp245', 'ssp585']
years = list(range(2024, 2101))

rows = []
for iso, region in country_region.items():
    if region not in ar6_regions:
        continue
    name = country_names.get(iso, iso)
    for scenario in scenarios:
        near, long_ = ar6_regions[region][scenario]
        for year in years:
            if year <= 2030:
                t = (year - 2024) / (2030 - 2024)
                # scale 2024 observed global (~1.2°C) by regional near/global ratio
                start = 1.2 * (near / 1.5)
                val = round(start + (near - start) * t, 3)
            elif year <= 2090:
                t = (year - 2030) / (2090 - 2030)
                val = round(near + (long_ - near) * t, 3)
            else:
                val = round(long_, 3)
            rows.append({'iso': iso, 'country': name, 'year': year,
                         'scenario': scenario, 'anomaly': val})

regional = pd.DataFrame(rows)
regional.to_csv('data/regional_warming.csv', index=False)
print(f"regional_warming.csv — {len(regional)} rows, {regional['iso'].nunique()} countries")
regional[regional['year'].isin([2030,2060,2090]) & (regional['scenario']=='ssp245')].groupby(['country','year'])['anomaly'].first().unstack().head(10)

## Done

Three files written to `data/`:
- `historical.csv` — NASA GISTEMP v4 annual anomaly vs 1850–1900 baseline
- `projections.csv` — IPCC AR6 SPM.1 SSP medians + likely ranges, annual 2024–2100
- `regional_warming.csv` — IPCC AR6 Atlas regional warming per country, annual 2024–2100

Copy these to `project3/data/` in your portfolio repo, then open `index.html`.